# 04 · Multi-head latent attention (MLA) and weight absorption

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/04_mla_absorption.ipynb) &nbsp; 📖 Concept deck: [MQA → GQA → MLA](https://nilesh-salpe.github.io/visual-ai/architecture/05-attention-variants/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** DeepSeek-V2's latent KV cache, the 'absorb W_UK into W_Q' trick that avoids decompressing keys, a demonstration that RoPE breaks the trick, and the decoupled-RoPE fix.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _rope_ref(x, pos, base=10000.0):
    """Interleaved-pair RoPE as in RoFormer: pairs (x0,x1), (x2,x3), ... rotate by pos * theta_i."""
    T, d = x.shape
    theta = base ** (-np.arange(0, d, 2) / d)            # [d/2]
    ang = np.asarray(pos, float)[:, None] * theta[None]  # [T, d/2]
    c, s = np.cos(ang), np.sin(ang)
    x0, x1 = x[:, 0::2], x[:, 1::2]
    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x0 * c - x1 * s
    out[:, 1::2] = x0 * s + x1 * c
    return out

r = np.random.default_rng(0)
d, n_h, d_h, d_c, d_R = 64, 4, 16, 8, 4          # model width, heads, head dim, latent dim, RoPE dim
W_DKV = r.normal(0, d ** -.5, (d, d_c))           # down-projection: x -> latent c (this is what gets cached)
W_UK = r.normal(0, d_c ** -.5, (n_h, d_c, d_h))   # per-head up-projection: c -> key
W_UV = r.normal(0, d_c ** -.5, (n_h, d_c, d_h))   # per-head up-projection: c -> value
W_Q = r.normal(0, d ** -.5, (n_h, d, d_h))        # per-head query projection
W_QR = r.normal(0, d ** -.5, (n_h, d, d_R))       # per-head RoPE query
W_KR = r.normal(0, d ** -.5, (d, d_R))            # ONE RoPE key shared by all heads
X = r.normal(size=(10, d))                        # 10 tokens

def _naive_scores(x_q, C):
    """Decompress every cached latent into full keys, then do ordinary attention scores (no RoPE)."""
    return np.stack([(x_q @ W_Q[h]) @ (C @ W_UK[h]).T for h in range(n_h)]) / np.sqrt(d_h)
print(f"per-token cache: MHA would store 2*n_h*d_h = {2 * n_h * d_h} numbers; MLA stores d_c + d_R = {d_c + d_R}")

## Task 1: Compress to the latent

MLA caches **one small latent vector per token**, $c_t = x_t W^{DKV}$ (width $d_c$), instead of $n_h$ keys and $n_h$ values.
Keys and values are reconstructed on demand: $k^{(h)}_t = c_t W^{UK}_h$ and $v^{(h)}_t = c_t W^{UV}_h$.

In [ ]:
def latent_cache(X):
    """X: [T, d]. Returns C: [T, d_c]."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_latent_cache(f):
    C = f(X)
    assert C.shape == (10, d_c), f"shape {C.shape} != (10, {d_c})"
    assert np.allclose(C, X @ W_DKV), "C must be X @ W_DKV"

check("latent_cache", _test_latent_cache, latent_cache)

## Task 2: Absorb W_UK into the query

Decompressing every past key at every step would waste the savings. Instead use associativity:
$$q^{(h)} k^{(h)\top}_t = (x W^Q_h)(c_t W^{UK}_h)^\top = x\,\underbrace{W^Q_h W^{UK\top}_h}_{\text{precompute: } [d,\ d_c]}\,c_t^\top$$
So the query is projected **straight into latent space** and scored against the cached $c_t$ directly. No key is ever materialized.
Return scores of shape [n_h, T_q, T_k] (scaled by $1/\sqrt{d_h}$, no masking).

In [ ]:
def absorbed_scores(x_q, C):
    """x_q: [T_q, d]; C: [T_k, d_c] cached latents. Returns [n_h, T_q, T_k] WITHOUT computing any key."""
    # TODO: for each head, W_absorbed = W_Q[h] @ W_UK[h].T  ([d, d_c]); scores = (x_q @ W_absorbed) @ C.T / sqrt(d_h)
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_absorbed_scores(f):
    C = X @ W_DKV; S = f(X[-3:], C)
    assert S.shape == (n_h, 3, 10), f"shape {S.shape} != ({n_h}, 3, 10)"
    assert np.allclose(S, _naive_scores(X[-3:], C), atol=1e-10), "absorbed scores must equal the decompress-then-score result exactly"

check("absorbed_scores", _test_absorbed_scores, absorbed_scores)

## Task 3: Why RoPE breaks absorption, and decoupled RoPE

With RoPE, the key becomes $R_t\,(c_t W^{UK}_h)$, where $R_t$ is a rotation that depends on position $t$. The middle matrix becomes $W^Q_h R_{m}^\top R_t W^{UK\top}_h$, which depends on **both positions**, so it can't be precomputed once. DeepSeek-V2: *"RoPE is incompatible with low-rank KV compression."*

**Fix (decoupled RoPE):** carry position in a separate small slice:
* content part: $q^{C}_h \cdot k^{C}_h$ via the absorbed latent path, with no RoPE;
* position part: $\mathrm{RoPE}_m(x W^{QR}_h) \cdot \mathrm{RoPE}_t(x_t W^{KR})$, where the RoPE key $k^R_t$ is **shared by all heads** and cached alongside $c_t$.

Score $= (q^C_h \cdot k^C_{h,t} + q^R_h \cdot k^R_t)/\sqrt{d_h + d_R}$. Implement it given cached `C` ([T, d_c]) and cached `KR` ([T, d_R], already rotated).

In [ ]:
def decoupled_rope_scores(x_q, q_pos, C, KR):
    """x_q: [T_q, d] at positions q_pos; C: [T_k, d_c]; KR: [T_k, d_R] rotated RoPE keys. Returns [n_h, T_q, T_k]."""
    # TODO: content = absorbed path (no RoPE); rope part = rope(x_q @ W_QR[h], q_pos) @ KR.T; sum, scale by 1/sqrt(d_h + d_R)
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_decoupled_rope_scores(f):
    pos = np.arange(10); C = X @ W_DKV; KR = _rope_ref(X @ W_KR, pos)
    S = f(X[-2:], pos[-2:], C, KR)
    assert S.shape == (n_h, 2, 10), f"shape {S.shape} != ({n_h}, 2, 10)"
    ref = np.stack([((X[-2:] @ W_Q[h]) @ (C @ W_UK[h]).T + _rope_ref(X[-2:] @ W_QR[h], pos[-2:]) @ KR.T) for h in range(n_h)]) / np.sqrt(d_h + d_R)
    assert np.allclose(S, ref, atol=1e-10), "scores differ from explicit content + RoPE computation"
    S2 = f(X[-2:], pos[-2:] + 500, C, _rope_ref(X @ W_KR, pos + 500))
    assert np.allclose(S, S2, atol=1e-10), "shifting all positions must not change scores (RoPE is relative)"

check("decoupled_rope_scores", _test_decoupled_rope_scores, decoupled_rope_scores)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Compress to the latent
def latent_cache_solution(X):
    return X @ W_DKV

check("latent_cache (reference)", _test_latent_cache, latent_cache_solution)

In [ ]:
# @title Solution for Task 2: Absorb W_UK into the query
def absorbed_scores_solution(x_q, C):
    return np.stack([(x_q @ (W_Q[h] @ W_UK[h].T)) @ C.T for h in range(n_h)]) / np.sqrt(d_h)

check("absorbed_scores (reference)", _test_absorbed_scores, absorbed_scores_solution)

In [ ]:
# @title Solution for Task 3: Why RoPE breaks absorption, and decoupled RoPE
def decoupled_rope_scores_solution(x_q, q_pos, C, KR):
    out = []
    for h in range(n_h):
        content = (x_q @ (W_Q[h] @ W_UK[h].T)) @ C.T
        pos_part = _rope_ref(x_q @ W_QR[h], q_pos) @ KR.T
        out.append(content + pos_part)
    return np.stack(out) / np.sqrt(d_h + d_R)

check("decoupled_rope_scores (reference)", _test_decoupled_rope_scores, decoupled_rope_scores_solution)

---
## Experiments
**1. See the incompatibility.** Apply RoPE directly to the decompressed keys, then try to reuse a single absorbed matrix. The error is far from zero.
**2. Cache sizes with DeepSeek-V2's real dimensions** ($n_h=128$, $d_h=128$, $d_c=512$, $d_R=64$; numbers cached per token per layer). The paper reports MLA ≈ GQA with about 2.25 groups in cache size, with stronger quality than MHA.

In [ ]:
C = X @ W_DKV; pos = np.arange(10)
true = np.stack([_rope_ref(X @ W_Q[h], pos) @ _rope_ref(C @ W_UK[h], pos).T for h in range(n_h)])
fake = np.stack([(X @ (W_Q[h] @ W_UK[h].T)) @ C.T for h in range(n_h)])
print(f"RoPE on full keys vs one precomputed absorbed matrix: max |error| = {np.abs(true - fake).max():.3f}  (not 0, so absorption fails)")
nh, dh, dc, dR = 128, 128, 512, 64
rows = [('MHA', 2 * nh * dh), ('GQA, 8 groups', 2 * 8 * dh), ('MQA', 2 * dh), ('MLA (d_c + d_R)', dc + dR)]
print('\nKV-cache numbers per token per layer (DeepSeek-V2 dims):')
for name, n in rows:
    print(f"  {name:16} {n:6,}   ({100 * n / rows[0][1]:5.1f}% of MHA)")
print(f"  MLA expressed in head-dims: {(dc + dR) / dh:.2f} * d_h  (paper: ~9/2 d_h)")

---
## Takeaways
* MQA and GQA shrink the cache by **sharing** K/V heads. MLA shrinks it by **compressing** all heads into one latent and re-expanding on the fly.
* Absorption ($W^QW^{UK\top}$) means the expansion never happens at inference. The query moves into latent space instead.
* RoPE's position-dependent rotation sits between the two matrices and breaks absorption, so DeepSeek-V2 adds a small separate RoPE slice ($d_R = 64$) with a key shared across heads.